# exp065 — R2 SC Pseudo Ablation (Colab Blackwell, single NB)

**Hypothesis**: exp058 R1 失敗の主因が「train_soundscapes 未利用」か検証

**Step 1 ablation**:
  exp058 R1 config そのまま + **SC pseudo 追加だけ**
  - Perch distill 追加なし (NO)
  - MixUp 追加なし (NO)
  - SC pseudo data だけ追加 (YES)

**Pipeline**:
  1. SC pseudo gen: exp020 R2 5-fold teacher で train_soundscapes (~10k files × 6 chunks)
  2. R2 training: BC2026 hard + XC pseudo (既存) + SC pseudo (新)
  3. R1 ckpt warm-start
  4. 15 epochs

**Output**: `maekeso/birdclef2026-exp065-effv2b0-r2-sc`

**Expected**:
  SC が真因なら: LB 0.85-0.90 (R1 0.76 から +0.09-0.14)
  SC で限定なら: LB 0.78-0.83 (Perch も必要)


In [1]:
!pip install -q timm==1.0.11 soundfile librosa kaggle 2>&1 | tail -1
from google.colab import drive
drive.mount('/content/drive')
import os
from pathlib import Path
DRIVE_ROOT = Path("/content/drive/MyDrive/kaggle/birdclef2026/output/exp065")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Drive: {DRIVE_ROOT}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 49.2 MB/s eta 0:00:00
Mounted at /content/drive
Drive: /content/drive/MyDrive/kaggle/birdclef2026/output/exp065


In [2]:
import os, json, time, shutil
from pathlib import Path

KAGGLE_DIR = Path.home() / ".kaggle"
KAGGLE_DIR.mkdir(exist_ok=True)
if not (KAGGLE_DIR / "kaggle.json").exists():
    src_kg = Path("/content/drive/MyDrive/kaggle/kaggle.json")
    if src_kg.exists():
        shutil.copy(src_kg, KAGGLE_DIR / "kaggle.json")
        os.chmod(KAGGLE_DIR / "kaggle.json", 0o600)
_kgat = json.loads((KAGGLE_DIR/"kaggle.json").read_text())["key"]
os.environ["KAGGLE_API_TOKEN"] = _kgat
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()
print("auth OK")

LOCAL_DATA = Path("/content/data")
LOCAL_DATA.mkdir(exist_ok=True)
t0 = time.time()

DATASETS = [
    "maekeso/birdclef2026-exp058-xc-pseudo",          # XC pseudo (既存)
    "maekeso/birdclef2026-xc-api-dl-part1",
    "maekeso/birdclef2026-xc-api-dl-part2",
    "maekeso/birdclef2026-xc-api-dl-part3",
    "maekeso/birdclef2026-exp020-weights-5fold",      # teacher for SC pseudo
    "maekeso/birdclef2026-exp058-effv2b0-combined",   # R1 ckpt for warm-start
]
for ds in DATASETS:
    name = ds.split("/")[-1]
    dst = LOCAL_DATA / name
    if dst.exists() and any(dst.iterdir()):
        n = sum(1 for _ in dst.rglob("*") if _.is_file())
        if n > 0: print(f"  ✓ {name} ({n} files)"); continue
    dst.mkdir(exist_ok=True)
    print(f"  DL {ds}...")
    api.dataset_download_files(ds, path=str(dst), unzip=True, quiet=False)

BC_DIR = LOCAL_DATA / "birdclef-2026"
if not (BC_DIR / "train.csv").exists():
    BC_DIR.mkdir(exist_ok=True)
    print("DL BC2026...")
    api.competition_download_files("birdclef-2026", path=str(BC_DIR), quiet=False)
    import zipfile
    zp = BC_DIR / "birdclef-2026.zip"
    if zp.exists():
        with zipfile.ZipFile(zp) as zf: zf.extractall(BC_DIR)
        zp.unlink()
print(f"DL: {(time.time()-t0)/60:.1f}min")


auth OK
  DL maekeso/birdclef2026-exp058-xc-pseudo...
Dataset URL: https://www.kaggle.com/datasets/maekeso/birdclef2026-exp058-xc-pseudo


100%|██████████| 320M/320M [00:02<00:00, 115MB/s] 



  DL maekeso/birdclef2026-xc-api-dl-part1...
Dataset URL: https://www.kaggle.com/datasets/maekeso/birdclef2026-xc-api-dl-part1


100%|██████████| 14.5G/14.5G [02:26<00:00, 106MB/s]



  DL maekeso/birdclef2026-xc-api-dl-part2...
Dataset URL: https://www.kaggle.com/datasets/maekeso/birdclef2026-xc-api-dl-part2


...


  DL maekeso/birdclef2026-exp058-effv2b0-combined...
Dataset URL: https://www.kaggle.com/datasets/maekeso/birdclef2026-exp058-effv2b0-combined


100%|██████████| 24.2M/24.2M [00:00<00:00, 90.9MB/s]



DL BC2026...


100%|██████████| 15.0G/15.0G [01:11<00:00, 224MB/s]



DL: 9.7min


In [3]:
import sys, gc, re, math, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torch.amp import autocast, GradScaler
import torchaudio
import soundfile as sf
import librosa
import timm
from tqdm.auto import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_float32_matmul_precision("high")
print(f"Device: {DEVICE}, mem: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB")
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)


Device: cuda, mem: 102.0GB


In [4]:
CFG = dict(
    backbone="tf_efficientnetv2_b0.in1k",
    in_chans=1,
    sr=32000, chunk_sec=5,
    n_mels=256, n_fft=2048, hop_length=512,
    fmin=20, fmax=16000,
    epochs=15,
    batch_size=192,
    lr=4e-4,                # ★ R1 lr 8e-4 の half (warm-start から再調整)
    weight_decay=1e-3,
    warmup_steps=300,       # ★ R1 500 → 300 (warm start なので shorter)
    label_smoothing=0.1,
    grad_clip=2.0,
    num_workers=8,
    val_split=0.1, val_seed=42,
    xc_pseudo_weight=0.5,
    sc_pseudo_weight=0.5,
    drop_path_rate=0.1,
    hidden_dim=512,
    use_mixup=False,
    spec_aug_freq_mask=30, spec_aug_time_mask=60,
    sc_max_chunks_per_file=6,  # match XC pattern, ~60k SC chunks (balanced)
)
for k, v in CFG.items(): print(f"  {k}: {v}")
CHUNK_SAMPLES = CFG["sr"] * CFG["chunk_sec"]
N_WINDOWS = 12


  backbone: tf_efficientnetv2_b0.in1k
  in_chans: 1
  sr: 32000
  chunk_sec: 5
  n_mels: 256
  n_fft: 2048
  hop_length: 512
  fmin: 20
  fmax: 16000
  epochs: 15
  batch_size: 192
  lr: 0.0004
  weight_decay: 0.001
  warmup_steps: 300
  label_smoothing: 0.1
  grad_clip: 2.0
  num_workers: 8
  val_split: 0.1
  val_seed: 42
  xc_pseudo_weight: 0.5
  sc_pseudo_weight: 0.5
  drop_path_rate: 0.1
  hidden_dim: 512
  use_mixup: False
  spec_aug_freq_mask: 30
  spec_aug_time_mask: 60
  sc_max_chunks_per_file: 6


In [5]:
__BC2026_SPECIES = [
    ('Guyalna cuta', '1161364', 'Insecta', 1161364),
    ('Caiman yacare', '116570', 'Reptilia', 116570),
    ('Leptodactylus luctator', '1176823', 'Amphibia', 1176823),
    ('Adenomera guarani', '1491113', 'Amphibia', 1491113),
    ('Lysapsus limellum', '1595929', 'Amphibia', 1595929),
    ('Equus caballus', '209233', 'Mammalia', 209233),
    ('Leptodactylus syphax', '22930', 'Amphibia', 22930),
    ('Leptodactylus mystacinus', '22956', 'Amphibia', 22956),
    ('Leptodactylus podicipinus', '22961', 'Amphibia', 22961),
    ('Leptodactylus elenae', '22967', 'Amphibia', 22967),
    ('Leptodactylus fuscus', '22973', 'Amphibia', 22973),
    ('Leptodactylus labyrinthicus', '22983', 'Amphibia', 22983),
    ('Leptodactylus petersii', '22985', 'Amphibia', 22985),
    ('Physalaemus centralis', '23150', 'Amphibia', 23150),
    ('Physalaemus albifrons', '23154', 'Amphibia', 23154),
    ('Physalaemus albonotatus', '23158', 'Amphibia', 23158),
    ('Pseudopaludicola mystacalis', '23176', 'Amphibia', 23176),
    ('Phyllomedusa sauvagii', '23724', 'Amphibia', 23724),
    ('Scinax nasicus', '24279', 'Amphibia', 24279),
    ('Scinax fuscovarius', '24285', 'Amphibia', 24285),
    ('Scinax fuscomarginatus', '24287', 'Amphibia', 24287),
    ('Scinax acuminatus', '24321', 'Amphibia', 24321),
    ('Quesada gigas', '244024', 'Insecta', 244024),
    ('Chiasmocleis mehelyi', '25073', 'Amphibia', 25073),
    ('Elachistocleis bicolor', '25092', 'Amphibia', 25092),
    ('Dermatonotus muelleri', '25214', 'Amphibia', 25214),
    ('Physalaemus biligonigerus', '326272', 'Amphibia', 326272),
    ('Panthera onca', '41970', 'Mammalia', 41970),
    ('Alouatta caraya', '43435', 'Mammalia', 43435),
    ('Canis familiaris', '47144', 'Mammalia', 47144),
    ('Insect son01', '47158son01', 'Insecta', 47158),
    ('Insect son02', '47158son02', 'Insecta', 47158),
    ('Insect son03', '47158son03', 'Insecta', 47158),
    ('Insect son04', '47158son04', 'Insecta', 47158),
    ('Insect son05', '47158son05', 'Insecta', 47158),
    ('Insect son06', '47158son06', 'Insecta', 47158),
    ('Insect son07', '47158son07', 'Insecta', 47158),
    ('Insect son08', '47158son08', 'Insecta', 47158),
    ('Insect son09', '47158son09', 'Insecta', 47158),
    ('Insect son10', '47158son10', 'Insecta', 47158),
    ('Insect son11', '47158son11', 'Insecta', 47158),
    ('Insect son12', '47158son12', 'Insecta', 47158),
    ('Insect son13', '47158son13', 'Insecta', 47158),
    ('Insect son14', '47158son14', 'Insecta', 47158),
    ('Insect son15', '47158son15', 'Insecta', 47158),
    ('Insect son16', '47158son16', 'Insecta', 47158),
    ('Insect son17', '47158son17', 'Insecta', 47158),
    ('Insect son18', '47158son18', 'Insecta', 47158),
    ('Insect son19', '47158son19', 'Insecta', 47158),
    ('Insect son20', '47158son20', 'Insecta', 47158),
    ('Insect son21', '47158son21', 'Insecta', 47158),
    ('Insect son22', '47158son22', 'Insecta', 47158),
    ('Insect son23', '47158son23', 'Insecta', 47158),
    ('Insect son24', '47158son24', 'Insecta', 47158),
    ('Insect son25', '47158son25', 'Insecta', 47158),
    ('Physalaemus nattereri', '476521', 'Amphibia', 476521),
    ('Sapajus cay', '516975', 'Mammalia', 516975),
    ('Pithecopus azureus', '517063', 'Amphibia', 517063),
    ('Boana lundii', '555123', 'Amphibia', 555123),
    ('Boana punctata', '555145', 'Amphibia', 555145),
    ('Boana raniceps', '555146', 'Amphibia', 555146),
    ('Ameerega picta', '64898', 'Amphibia', 64898),
    ('Dendropsophus minutus', '65377', 'Amphibia', 65377),
    ('Dendropsophus nanus', '65380', 'Amphibia', 65380),
    ('Pseudis platensis', '66971', 'Amphibia', 66971),
    ('Rhinella diptycha', '67107', 'Amphibia', 67107),
    ('Trachycephalus typhonius', '67252', 'Amphibia', 67252),
    ('Leptodactylus macrosternum', '70711', 'Amphibia', 70711),
    ('Plecturocebus pallescens', '738183', 'Mammalia', 738183),
    ('Bos taurus', '74113', 'Mammalia', 74113),
    ('Mico melanurus', '74580', 'Mammalia', 74580),
    ('Prionacris erosa', '760266', 'Insecta', 760266),
    ('Hylophilus pectoralis', 'ashgre1', 'Aves', 17431),
    ('Mustelirallus albicollis', 'astcra1', 'Aves', 508907),
    ('Crax fasciolata', 'bafcur1', 'Aves', 2046),
    ('Micrastur ruficollis', 'baffal1', 'Aves', 4699),
    ('Coereba flaveola', 'banana', 'Aves', 10199),
    ('Thamnophilus doliatus', 'barant1', 'Aves', 15764),
    ('Procnias nudicollis', 'batbel1', 'Aves', 8854),
    ('Ara ararauna', 'baymac', 'Aves', 19018),
    ('Dendrocygna autumnalis', 'bbwduc', 'Aves', 6893),
    ('Microspingus melanoleucus', 'bcwfin2', 'Aves', 558564),
    ('Donacobius atricapilla', 'bkcdon', 'Aves', 116877),
    ('Aratinga nenday', 'bkhpar', 'Aves', 367562),
    ('Busarellus nigricollis', 'blchaw1', 'Aves', 5346),
    ('Spizaetus tyrannus', 'blheag1', 'Aves', 5291),
    ('Tityra cayana', 'blttit1', 'Aves', 8830),
    ('Myiarchus tyrannulus', 'bncfly', 'Aves', 16016),
    ('Megarynchus pitangua', 'bobfly1', 'Aves', 16737),
    ('Progne tapera', 'brcmar1', 'Aves', 11870),
    ('Tyto furcata', 'brnowl', 'Aves', 1578502),
    ('Momotus momota', 'bucmot4', 'Aves', 204447),
    ('Thectocercus acuticaudatus', 'bucpar', 'Aves', 367564),
    ('Amazona aestiva', 'bufpar', 'Aves', 18978),
    ('Theristicus caudatus', 'bunibi1', 'Aves', 3766),
    ('Athene cunicularia', 'burowl', 'Aves', 19975),
    ('Colaptes campestris', 'camfli1', 'Aves', 18262),
    ('Ortalis canicollis', 'chacha1', 'Aves', 2088),
    ('Mimus saturninus', 'chbmoc1', 'Aves', 14878),
    ('Gnorimopsar chopi', 'chobla1', 'Aves', 10723),
    ('Conirostrum speciosum', 'chvcon1', 'Aves', 10013),
    ('Synallaxis hypospodia', 'cibspi1', 'Aves', 10921),
    ('Micrastur semitorquatus', 'coffal1', 'Aves', 4698),
    ('Nyctidromus albicollis', 'compau', 'Aves', 19627),
    ('Nyctibius griseus', 'compot1', 'Aves', 19667),
    ('Turdus amaurochalinus', 'crbthr1', 'Aves', 12710),
    ('Pachyramphus validus', 'crebec1', 'Aves', 8681),
    ('Taoniscus nanus', 'dwatin1', 'Aves', 20714),
    ('Icterus pyrrhopterus', 'epaori4', 'Aves', 72954),
    ('Lathrotriccus euleri', 'eulfly1', 'Aves', 17231),
    ('Cantorchilus guarayanus', 'fabwre1', 'Aves', 144880),
    ('Glaucidium brasilianum', 'fepowl', 'Aves', 19822),
    ('Machaeropterus pyrocephalus', 'ficman1', 'Aves', 14344),
    ('Myiothlypis flaveola', 'flawar1', 'Aves', 201223),
    ('Tyrannus savana', 'fotfly', 'Aves', 16793),
    ('Cnemotriccus fuscatus', 'fusfly1', 'Aves', 17076),
    ('Hylocharis chrysura', 'gilhum1', 'Aves', 5979),
    ('Aramides ypecaha', 'giwrai1', 'Aves', 460),
    ('Chionomesa fimbriata', 'glteme1', 'Aves', 1289639),
    ('Saltator coerulescens', 'grasal3', 'Aves', 9850),
    ('Crotophaga major', 'greani1', 'Aves', 1970),
    ('Taraba major', 'greant1', 'Aves', 15957),
    ('Myiopagis viridicata', 'greela', 'Aves', 16892),
    ('Pitangus sulphuratus', 'grekis', 'Aves', 16956),
    ('Nyctibius grandis', 'grepot1', 'Aves', 19680),
    ('Phacellodomus ruber', 'gretho2', 'Aves', 11632),
    ('Tringa melanoleuca', 'greyel', 'Aves', 3892),
    ('Leptotila rufaxilla', 'grfdov1', 'Aves', 3302),
    ('Eucometis penicillata', 'grhtan1', 'Aves', 10698),
    ('Aramides cajaneus', 'gycwor1', 'Aves', 513889),
    ('Anhima cornuta', 'horscr1', 'Aves', 6908),
    ('Passer domesticus', 'houspa', 'Aves', 13858),
    ('Anodorhynchus hyacinthinus', 'hyamac1', 'Aves', 18938),
    ('Elaenia spectabilis', 'larela1', 'Aves', 16734),
    ('Elaenia chiriquensis', 'lesela1', 'Aves', 578460),
    ('Emberizoides ypiranganus', 'lesgrf1', 'Aves', 10555),
    ('Aramus guarauna', 'limpki', 'Aves', 7),
    ('Dryocopus lineatus', 'linwoo1', 'Aves', 17858),
    ('Coccycua minuta', 'litcuc2', 'Aves', 72740),
    ('Setopagis parvula', 'litnig1', 'Aves', 367507),
    ('Pyrrhura frontalis', 'mabpar', 'Aves', 19162),
    ('Cercomacra melanaria', 'magant1', 'Aves', 15737),
    ('Cissopis leverianus', 'magtan2', 'Aves', 72727),
    ('Polioptila dumicola', 'masgna1', 'Aves', 7509),
    ('Chordeiles nacunda', 'nacnig1', 'Aves', 19661),
    ('Rufirallus schomburgkii', 'ocecra1', 'Aves', 1506288),
    ('Sittasomus griseicapillus', 'oliwoo1', 'Aves', 11511),
    ('Icterus croconotus', 'orbtro3', 'Aves', 62564),
    ('Amazona amazonica', 'orwpar', 'Aves', 18982),
    ('Pandion haliaetus', 'osprey', 'Aves', 116999),
    ('Synallaxis albescens', 'pabspi1', 'Aves', 10999),
    ('Furnarius leucopus', 'palhor3', 'Aves', 11281),
    ('Thraupis palmarum', 'paltan1', 'Aves', 10297),
    ('Dromococcyx phasianellus', 'phecuc1', 'Aves', 1982),
    ('Patagioenas picazuro', 'picpig2', 'Aves', 3102),
    ('Legatus leucophaius', 'pirfly1', 'Aves', 17312),
    ('Thamnophilus pelzelni', 'plasla1', 'Aves', 73493),
    ('Inezia inornata', 'platyr1', 'Aves', 16344),
    ('Cyanocorax chrysops', 'plcjay1', 'Aves', 8484),
    ('Theristicus caerulescens', 'pluibi1', 'Aves', 3768),
    ('Cyanocorax cyanomelas', 'purjay1', 'Aves', 8483),
    ('Hemitriccus margaritaceiventer', 'pvttyr1', 'Aves', 16273),
    ('Ara chloropterus', 'ragmac1', 'Aves', 19016),
    ('Campylorhamphus trochilirostris', 'rebscy1', 'Aves', 11201),
    ('Coryphospingus cucullatus', 'recfin1', 'Aves', 10310),
    ('Gallus gallus', 'redjun', 'Aves', 882),
    ('Cariama cristata', 'relser1', 'Aves', 14),
    ('Megaceryle torquata', 'rinkin1', 'Aves', 2552),
    ('Myiothlypis rivularis', 'rivwar1', 'Aves', 145267),
    ('Rupornis magnirostris', 'roahaw', 'Aves', 201041),
    ('Turdus rufiventris', 'rubthr1', 'Aves', 12738),
    ('Pseudoseisura unirufa', 'rufcac2', 'Aves', 11718),
    ('Casiornis rufus', 'rufcas2', 'Aves', 17102),
    ('Conopophaga lineata', 'rufgna3', 'Aves', 578313),
    ('Furnarius rufus', 'rufhor2', 'Aves', 11275),
    ('Antrostomus rufus', 'rufnig1', 'Aves', 201066),
    ('Phacellodomus rufifrons', 'ruftho1', 'Aves', 11624),
    ('Poecilotriccus latirostris', 'ruftof1', 'Aves', 17026),
    ('Myiozetetes cayanensis', 'rumfly1', 'Aves', 16833),
    ('Tigrisoma lineatum', 'ruther1', 'Aves', 5048),
    ('Galbula ruficauda', 'rutjac1', 'Aves', 1468),
    ('Arremon flavirostris', 'sabspa1', 'Aves', 10064),
    ('Sicalis flaveola', 'saffin', 'Aves', 9864),
    ('Thraupis sayaca', 'saytan1', 'Aves', 10293),
    ('Columbina squammata', 'scadov1', 'Aves', 3564),
    ('Pionus maximiliani', 'schpar1', 'Aves', 19094),
    ('Phaethornis eurynome', 'scther1', 'Aves', 5622),
    ('Myiarchus ferox', 'shcfly1', 'Aves', 16006),
    ('Accipiter striatus', 'shshaw', 'Aves', 5097),
    ('Lurocalis semitorquatus', 'shtnig1', 'Aves', 19645),
    ('Ramphocelus carbo', 'sibtan2', 'Aves', 10056),
    ('Crotophaga ani', 'smbani', 'Aves', 1971),
    ('Crypturellus parvirostris', 'smbtin1', 'Aves', 20570),
    ('Cacicus solitarius', 'sobcac1', 'Aves', 10365),
    ('Camptostoma obsoletum', 'sobtyr1', 'Aves', 16972),
    ('Myiozetetes similis', 'socfly1', 'Aves', 16842),
    ('Synallaxis frontalis', 'sofspi1', 'Aves', 10996),
    ('Corythopis delalandi', 'souant1', 'Aves', 17264),
    ('Vanellus chilensis', 'soulap1', 'Aves', 4867),
    ('Chauna torquata', 'souscr1', 'Aves', 6910),
    ('Hypoedaleus guttatus', 'spbant3', 'Aves', 15959),
    ('Synallaxis spixi', 'spispi1', 'Aves', 10915),
    ('Antiurus maculicaudus', 'sptnig1', 'Aves', 1584760),
    ('Piaya cayana', 'squcuc1', 'Aves', 1758),
    ('Dendroplex picus', 'stbwoo2', 'Aves', 72806),
    ('Tapera naevia', 'strcuc1', 'Aves', 1989),
    ('Butorides striata', 'strher2', 'Aves', 62528),
    ('Asio clamator', 'strowl1', 'Aves', 558468),
    ('Eupetomena macroura', 'swthum1', 'Aves', 6065),
    ('Chiroxiphia caudata', 'swtman1', 'Aves', 14306),
    ('Crypturellus tataupa', 'tattin1', 'Aves', 20587),
    ('Campylorhynchus turdinus', 'thlwre1', 'Aves', 7480),
    ('Ramphastos toco', 'toctou1', 'Aves', 18793),
    ('Tyrannus melancholicus', 'trokin', 'Aves', 16787),
    ('Megascops choliba', 'trsowl', 'Aves', 19788),
    ('Crypturellus undulatus', 'undtin1', 'Aves', 20592),
    ('Thamnophilus caerulescens', 'varant1', 'Aves', 15757),
    ('Jacana jacana', 'watjac1', 'Aves', 4580),
    ('Pyriglena maura', 'wesfie1', 'Aves', 1286886),
    ('Dendrocygna viduata', 'wfwduc1', 'Aves', 6898),
    ('Biatas nigropectus', 'whbant2', 'Aves', 15902),
    ('Myiothlypis leucoblephara', 'whbwar2', 'Aves', 201224),
    ('Melanerpes candidus', 'whiwoo1', 'Aves', 18183),
    ('Synallaxis albilora', 'whlspi1', 'Aves', 10992),
    ('Cyanocorax cyanopogon', 'whnjay1', 'Aves', 8469),
    ('Leptotila verreauxi', 'whtdov', 'Aves', 3280),
    ('Picumnus albosquamatus', 'whwpic1', 'Aves', 17786),
    ('Caracara plancus', 'y00678', 'Aves', 4715),
    ('Paroaria capitata', 'yebcar', 'Aves', 10257),
    ('Elaenia flavogaster', 'yebela1', 'Aves', 16714),
    ('Primolius auricollis', 'yecmac', 'Aves', 73272),
    ('Brotogeris chiriri', 'yecpar', 'Aves', 19215),
    ('Daptrius chimachima', 'yehcar1', 'Aves', 1432779),
    ('Tolmomyias sulphurescens', 'yeofly1', 'Aves', 16567),
]
species_df = pd.DataFrame(__BC2026_SPECIES, columns=["scientific_name", "primary_label", "class_name", "inat_taxon_id"])
PRIMARY_LABELS = species_df["primary_label"].tolist()
N_CLASSES = len(PRIMARY_LABELS)
label_to_idx = {l: i for i, l in enumerate(PRIMARY_LABELS)}
LABEL_TO_CLASS = dict(zip(species_df['primary_label'], species_df['class_name']))
print(f"BC2026: {N_CLASSES} species")


BC2026: 234 species


In [6]:
# Teacher: exp020 R2 5-fold for SC pseudo gen
E20_DIR = next(Path("/content/data/birdclef2026-exp020-weights-5fold").rglob("r2_fold0_ckpt_best_ns22.pth")).parent
print(f"teacher dir: {E20_DIR}")

class _GeMFreq(nn.Module):
    def __init__(self, p_init=3.0, eps=1e-6):
        super().__init__(); self.p = nn.Parameter(torch.tensor(float(p_init))); self.eps = eps
    def forward(self, x):
        p = self.p.clamp(min=1.0); x = x.clamp(min=self.eps).pow(p)
        return x.mean(dim=2).pow(1.0 / p)

class _DistillHead(nn.Module):
    def __init__(self, bd, ed=1536):
        super().__init__(); self.proj = nn.Linear(bd, ed)
    def forward(self, fm): return self.proj(fm.mean(dim=[2,3]))

class _E20SED(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model("eca_nfnet_l0", pretrained=False, in_chans=1, num_classes=0, global_pool="", drop_path_rate=0.1)
        with torch.no_grad():
            n_tf = CHUNK_SAMPLES // CFG["hop_length"] + 1
            dummy = torch.randn(1, 1, CFG["n_mels"], n_tf)
            self.backbone_dim = self.backbone(dummy).shape[1]
        self.gem_freq = _GeMFreq(3.0)
        self.dense = nn.Sequential(nn.Dropout(0.25), nn.Linear(self.backbone_dim, 512), nn.ReLU(inplace=True), nn.Dropout(0.5))
        self.att = nn.Conv1d(512, N_CLASSES, kernel_size=1, bias=True)
        self.cla = nn.Conv1d(512, N_CLASSES, kernel_size=1, bias=True)
        self.distill_head = _DistillHead(self.backbone_dim, 1536)
    def forward(self, x):
        h = self.backbone(x); h_cls = h.detach()
        h_cls = self.gem_freq(h_cls).permute(0, 2, 1)
        h_cls = self.dense(h_cls).permute(0, 2, 1)
        norm_att = torch.softmax(torch.tanh(self.att(h_cls)), dim=-1)
        fw = self.cla(h_cls)
        clip = torch.sum(norm_att * fw, dim=2)
        return clip, fw.permute(0, 2, 1)

class _MelTF(nn.Module):
    def __init__(self):
        super().__init__()
        self.mel = torchaudio.transforms.MelSpectrogram(CFG["sr"], n_fft=CFG["n_fft"], hop_length=CFG["hop_length"], n_mels=CFG["n_mels"], f_min=CFG["fmin"], f_max=CFG["fmax"], power=2.0)
        self.db = torchaudio.transforms.AmplitudeToDB(top_db=80)
    def forward(self, x): return self.db(self.mel(x))

teacher_mel_tf = _MelTF().to(DEVICE)
teacher_ckpts = sorted(E20_DIR.rglob("r2_fold*_ckpt_best_ns22.pth"))
teacher_models = []
for ck in teacher_ckpts:
    try: st = torch.load(str(ck), map_location="cpu", weights_only=False)
    except TypeError: st = torch.load(str(ck), map_location="cpu")
    m = _E20SED().to(DEVICE); m.load_state_dict(st["model_state"], strict=False); m.eval()
    teacher_models.append(m); del st; gc.collect()
print(f"Loaded {len(teacher_models)} teacher folds")


teacher dir: /content/data/birdclef2026-exp020-weights-5fold
Loaded 5 teacher folds


In [7]:
# Generate SC pseudo with teacher (5-fold avg)
from scipy.ndimage import gaussian_filter1d
SC_DIR = Path("/content/data/birdclef-2026/train_soundscapes")
sc_files = sorted(SC_DIR.glob("*.ogg"))
print(f"SC files: {len(sc_files)}")

# Sample SC chunks: random sc_max_chunks_per_file per file (deterministic seed)
SC_RNG = np.random.RandomState(42)
sc_entries = []  # (file_idx, chunk_idx)
for fi in range(len(sc_files)):
    chunks = SC_RNG.choice(N_WINDOWS, size=min(N_WINDOWS, CFG["sc_max_chunks_per_file"]), replace=False)
    for ci in sorted(chunks):
        sc_entries.append((fi, int(ci)))
print(f"SC entries (chunks to process): {len(sc_entries)}")

class _SCChunkDataset(Dataset):
    def __init__(self, files, entries):
        self.files = files; self.entries = entries
    def __len__(self): return len(self.entries)
    def __getitem__(self, idx):
        fi, ci = self.entries[idx]
        fp = self.files[fi]
        try:
            wav, sr = sf.read(str(fp), dtype="float32")
            if wav.ndim > 1: wav = wav.mean(axis=1)
            if sr != CFG["sr"]:
                wav = librosa.resample(wav, orig_sr=sr, target_sr=CFG["sr"])
        except Exception:
            wav = np.zeros(60 * CFG["sr"], dtype=np.float32)
        target_len = 60 * CFG["sr"]
        if len(wav) < target_len: wav = np.pad(wav, (0, target_len - len(wav)))
        else: wav = wav[:target_len]
        chunk = wav[ci*CHUNK_SAMPLES:(ci+1)*CHUNK_SAMPLES]
        return torch.from_numpy(chunk).float(), fi, ci

sc_ds_gen = _SCChunkDataset(sc_files, sc_entries)
sc_loader_gen = DataLoader(sc_ds_gen, batch_size=64, shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True)

# Determine T_frames
with torch.no_grad():
    w, _, _ = next(iter(sc_loader_gen))
    w = w.to(DEVICE).unsqueeze(1)
    m = teacher_mel_tf(w); m = (m - m.mean()) / (m.std() + 1e-6)
    _, fw = teacher_models[0](m)
    T_TEACHER = fw.shape[1]
print(f"T_TEACHER: {T_TEACHER}")

sc_pseudo_arr = np.zeros((len(sc_ds_gen), T_TEACHER, N_CLASSES), dtype=np.float16)
print(f"SC pseudo array size: {sc_pseudo_arr.nbytes/1e9:.2f}GB")

t0 = time.time()
chunk_g = 0
for bi, (wavs, fis, cis) in enumerate(sc_loader_gen):
    wavs = wavs.to(DEVICE, non_blocking=True).unsqueeze(1)
    with torch.no_grad():
        mel = teacher_mel_tf(wavs)
        mel = (mel - mel.mean(dim=(2, 3), keepdim=True)) / (mel.std(dim=(2, 3), keepdim=True) + 1e-6)
        accum = None
        for m in teacher_models:
            _, fw = m(mel)
            fw_prob = torch.sigmoid(fw)
            accum = fw_prob if accum is None else accum + fw_prob
        accum = (accum / len(teacher_models)).float().cpu().numpy().astype(np.float16)
    bsz = accum.shape[0]
    sc_pseudo_arr[chunk_g:chunk_g + bsz] = accum
    chunk_g += bsz
    if (bi + 1) % 50 == 0:
        el = (time.time() - t0) / 60
        eta = el / (bi + 1) * (len(sc_loader_gen) - bi - 1)
        print(f"  SC pseudo [{bi+1}/{len(sc_loader_gen)}] el={el:.1f}min eta={eta:.1f}min")

print(f"SC pseudo done: {sc_pseudo_arr.shape}, mean={sc_pseudo_arr.mean():.4f} in {(time.time()-t0)/60:.1f}min")

# Free teacher
del teacher_models, teacher_mel_tf
gc.collect()
torch.cuda.empty_cache()


SC files: 10658
SC entries (chunks to process): 63948
T_TEACHER: 10
SC pseudo array size: 0.30GB
  SC pseudo [50/1000] el=0.4min eta=8.2min
  SC pseudo [100/1000] el=0.8min eta=7.3min
  SC pseudo [150/1000] el=1.2min eta=6.8min
  SC pseudo [200/1000] el=1.6min eta=6.3min
  SC pseudo [250/1000] el=2.0min eta=5.9min
  SC pseudo [300/1000] el=2.4min eta=5.5min
  SC pseudo [350/1000] el=2.7min eta=5.1min
  SC pseudo [400/1000] el=3.1min eta=4.7min
  SC pseudo [450/1000] el=3.5min eta=4.3min
  SC pseudo [500/1000] el=3.9min eta=3.9min
  SC pseudo [550/1000] el=4.3min eta=3.5min
  SC pseudo [600/1000] el=4.7min eta=3.1min
  SC pseudo [650/1000] el=5.1min eta=2.7min
  SC pseudo [700/1000] el=5.5min eta=2.4min
  SC pseudo [750/1000] el=5.9min eta=2.0min
  SC pseudo [800/1000] el=6.3min eta=1.6min
  SC pseudo [850/1000] el=6.7min eta=1.2min
  SC pseudo [900/1000] el=7.1min eta=0.8min
  SC pseudo [950/1000] el=7.5min eta=0.4min
  SC pseudo [1000/1000] el=7.9min eta=0.0min
SC pseudo done: (63948,

In [8]:
# Load existing XC pseudo
PSEUDO_DIR = Path("/content/data/birdclef2026-exp058-xc-pseudo")
xc_npz = next(PSEUDO_DIR.rglob("xc_pseudo.npz"))
xc_idx_csv = next(PSEUDO_DIR.rglob("xc_pseudo_index.csv"))
print(f"loading XC: {xc_npz}")
xc_pseudo_arr = np.load(xc_npz)["pseudo"]
xc_index = pd.read_csv(xc_idx_csv)
print(f"XC pseudo: {xc_pseudo_arr.shape}, index: {len(xc_index)} rows")


loading XC: /content/data/birdclef2026-exp058-xc-pseudo/xc_pseudo.npz
XC pseudo: (79902, 10, 234), index: 79902 rows


In [9]:
# BC2026 train_audio metadata + val split
BC_DIR = Path("/content/data/birdclef-2026")
train_csv = pd.read_csv(BC_DIR / "train.csv")
print(f"train.csv: {len(train_csv)} rows")

train_audio_dir = BC_DIR / "train_audio"
bc_records = []
for _, r in train_csv.iterrows():
    pl = str(r["primary_label"])
    if pl not in label_to_idx: continue
    fp = train_audio_dir / str(r["filename"])
    if fp.exists(): bc_records.append((str(fp), pl))

bc_df = pd.DataFrame(bc_records, columns=["filepath", "primary_label"])
bc_df["target_idx"] = bc_df["primary_label"].map(label_to_idx)

np.random.seed(CFG["val_seed"])
bc_df = bc_df.sample(frac=1, random_state=CFG["val_seed"]).reset_index(drop=True)
val_mask = np.zeros(len(bc_df), dtype=bool)
for sp in bc_df["primary_label"].unique():
    idx = bc_df.index[bc_df["primary_label"] == sp].tolist()
    n_val = max(2, int(len(idx) * CFG["val_split"]))
    pick = np.random.choice(idx, size=min(n_val, len(idx)), replace=False)
    val_mask[pick] = True
bc_train_df = bc_df[~val_mask].reset_index(drop=True)
bc_val_df = bc_df[val_mask].reset_index(drop=True)
print(f"BC train: {len(bc_train_df)} | val: {len(bc_val_df)}")


train.csv: 35549 rows
BC train: 32035 | val: 3514


In [10]:
class BC2026Dataset(Dataset):
    def __init__(self, df, training=True):
        self.df = df.reset_index(drop=True); self.training = training
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]; fp = row["filepath"]; target_idx = int(row["target_idx"])
        try:
            wav, sr = sf.read(fp, dtype="float32")
            if wav.ndim > 1: wav = wav.mean(axis=1)
            if sr != CFG["sr"]:
                wav = librosa.resample(wav, orig_sr=sr, target_sr=CFG["sr"])
        except Exception:
            wav = np.zeros(CHUNK_SAMPLES, dtype=np.float32)
        if len(wav) < CHUNK_SAMPLES: wav = np.pad(wav, (0, CHUNK_SAMPLES - len(wav)))
        else:
            start = np.random.randint(0, len(wav) - CHUNK_SAMPLES + 1) if self.training else (len(wav) - CHUNK_SAMPLES) // 2
            wav = wav[start:start + CHUNK_SAMPLES]
        return {
            "wav": torch.from_numpy(wav).float(),
            "clip_target_idx": target_idx,
            "frame_pseudo": torch.zeros(1, dtype=torch.float16),
            "source_flag": 0,  # BC
        }


class XCPseudoDataset(Dataset):
    def __init__(self, idx_df, pseudo_arr):
        self.idx_df = idx_df.reset_index(drop=True); self.pseudo = pseudo_arr
        self.chunk_stride_sec = 5
    def __len__(self): return len(self.idx_df)
    def __getitem__(self, idx):
        row = self.idx_df.iloc[idx]; fp = row["filepath"]; ci = int(row["chunk_idx"])
        gidx = int(row["chunk_global_idx"])
        try:
            wav, sr = sf.read(fp, dtype="float32")
            if wav.ndim > 1: wav = wav.mean(axis=1)
            if sr != CFG["sr"]:
                wav = librosa.resample(wav, orig_sr=sr, target_sr=CFG["sr"])
        except Exception:
            wav = np.zeros(CHUNK_SAMPLES, dtype=np.float32)
        ss = ci * CFG["sr"] * self.chunk_stride_sec; es = ss + CHUNK_SAMPLES
        if es > len(wav):
            wav_chunk = np.zeros(CHUNK_SAMPLES, dtype=np.float32)
            avail = min(CHUNK_SAMPLES, max(0, len(wav) - ss))
            if avail > 0: wav_chunk[:avail] = wav[ss:ss + avail]
        else:
            wav_chunk = wav[ss:es]
        pseudo = self.pseudo[gidx]
        return {
            "wav": torch.from_numpy(wav_chunk).float(),
            "clip_target_idx": -1,
            "frame_pseudo": torch.from_numpy(pseudo),
            "source_flag": 1,  # XC
        }


class SCPseudoDataset(Dataset):
    def __init__(self, files, entries, pseudo_arr):
        self.files = files; self.entries = entries; self.pseudo = pseudo_arr
    def __len__(self): return len(self.entries)
    def __getitem__(self, idx):
        fi, ci = self.entries[idx]; fp = self.files[fi]
        try:
            wav, sr = sf.read(str(fp), dtype="float32")
            if wav.ndim > 1: wav = wav.mean(axis=1)
            if sr != CFG["sr"]:
                wav = librosa.resample(wav, orig_sr=sr, target_sr=CFG["sr"])
        except Exception:
            wav = np.zeros(60 * CFG["sr"], dtype=np.float32)
        target_len = 60 * CFG["sr"]
        if len(wav) < target_len: wav = np.pad(wav, (0, target_len - len(wav)))
        else: wav = wav[:target_len]
        chunk = wav[ci*CHUNK_SAMPLES:(ci+1)*CHUNK_SAMPLES]
        return {
            "wav": torch.from_numpy(chunk).float(),
            "clip_target_idx": -1,
            "frame_pseudo": torch.from_numpy(self.pseudo[idx]),
            "source_flag": 2,  # SC
        }


bc_train_ds = BC2026Dataset(bc_train_df, training=True)
bc_val_ds = BC2026Dataset(bc_val_df, training=False)
xc_ds = XCPseudoDataset(xc_index, xc_pseudo_arr)
sc_ds = SCPseudoDataset(sc_files, sc_entries, sc_pseudo_arr)

combined_train_ds = ConcatDataset([bc_train_ds, xc_ds, sc_ds])
print(f"BC train: {len(bc_train_ds)} | XC: {len(xc_ds)} | SC: {len(sc_ds)} | combined: {len(combined_train_ds)}")
print(f"Val (BC only): {len(bc_val_ds)}")


BC train: 32035 | XC: 79902 | SC: 63948 | combined: 175885
Val (BC only): 3514


In [11]:
class MelSpecTransform(nn.Module):
    def __init__(self):
        super().__init__()
        self.mel_spec = torchaudio.transforms.MelSpectrogram(CFG["sr"], n_fft=CFG["n_fft"], hop_length=CFG["hop_length"], n_mels=CFG["n_mels"], f_min=CFG["fmin"], f_max=CFG["fmax"], power=2.0)
        self.db = torchaudio.transforms.AmplitudeToDB(top_db=80)
    def forward(self, x): return self.db(self.mel_spec(x))


class GeMFreqPool(nn.Module):
    def __init__(self, p_init=3.0, eps=1e-6):
        super().__init__(); self.p = nn.Parameter(torch.tensor(float(p_init))); self.eps = eps
    def forward(self, x):
        p = self.p.clamp(min=1.0); x = x.clamp(min=self.eps).pow(p)
        return x.mean(dim=2).pow(1.0 / p)


class BirdSEDModelEffv2(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model(CFG["backbone"], pretrained=True, in_chans=CFG["in_chans"], num_classes=0, global_pool="", drop_path_rate=CFG["drop_path_rate"])
        with torch.no_grad():
            n_tf = CHUNK_SAMPLES // CFG["hop_length"] + 1
            dummy = torch.randn(1, CFG["in_chans"], CFG["n_mels"], n_tf)
            self.backbone_dim = self.backbone(dummy).shape[1]
        self.gem_freq = GeMFreqPool(3.0)
        self.dense = nn.Sequential(nn.Dropout(0.25), nn.Linear(self.backbone_dim, CFG["hidden_dim"]), nn.ReLU(inplace=True), nn.Dropout(0.5))
        self.att = nn.Conv1d(CFG["hidden_dim"], N_CLASSES, kernel_size=1, bias=True)
        self.cla = nn.Conv1d(CFG["hidden_dim"], N_CLASSES, kernel_size=1, bias=True)
    def forward(self, x, return_framewise=False):
        h = self.backbone(x)
        h_cls = self.gem_freq(h).permute(0, 2, 1)
        h_cls = self.dense(h_cls).permute(0, 2, 1)
        norm_att = torch.softmax(torch.tanh(self.att(h_cls)), dim=-1)
        fw = self.cla(h_cls)
        clip = torch.sum(norm_att * fw, dim=2)
        if return_framewise: return clip, fw.permute(0, 2, 1)
        return clip


mel_extractor = MelSpecTransform().to(DEVICE)
model = BirdSEDModelEffv2().to(DEVICE)

# WARM-START from exp058 R1
R1_CKPT_PATH = next(Path("/content/data/birdclef2026-exp058-effv2b0-combined").rglob("effv2b0_combined_best.pth"))
print(f"R1 ckpt: {R1_CKPT_PATH}")
try: r1_state = torch.load(str(R1_CKPT_PATH), map_location="cpu", weights_only=False)
except TypeError: r1_state = torch.load(str(R1_CKPT_PATH), map_location="cpu")
r1_sd = r1_state.get("state_dict", r1_state)
miss, unexp = model.load_state_dict(r1_sd, strict=False)
print(f"R1 warm-start: missing={len(miss)}, unexpected={len(unexp)}")
print(f"R1 val_ns22={r1_state.get('val_ns22'):.4f}, val_macro={r1_state.get('val_macro'):.4f}")
print(f"Student params: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")

# Determine T_student
with torch.no_grad():
    dw = torch.randn(1, CHUNK_SAMPLES, device=DEVICE)
    dm = mel_extractor(dw.unsqueeze(1))
    _, df_ = model(dm, return_framewise=True)
    T_STUDENT = df_.shape[1]
print(f"T_STUDENT: {T_STUDENT}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/28.8M [00:00<?, ?B/s]

R1 ckpt: /content/data/birdclef2026-exp058-effv2b0-combined/effv2b0_combined_best.pth
R1 warm-start: missing=0, unexpected=0
R1 val_ns22=0.8341, val_macro=0.9678
Student params: 6.75M
T_STUDENT: 10


In [12]:
def spec_augment(mel, freq_mask=CFG["spec_aug_freq_mask"], time_mask=CFG["spec_aug_time_mask"], n_freq=2, n_time=2):
    B, _, F_, T = mel.shape
    for _ in range(n_freq):
        f = np.random.randint(0, freq_mask + 1); f0 = np.random.randint(0, max(1, F_ - f))
        mel[:, :, f0:f0+f, :] = 0
    for _ in range(n_time):
        t = np.random.randint(0, time_mask + 1); t0 = np.random.randint(0, max(1, T - t))
        mel[:, :, :, t0:t0+t] = 0
    return mel


In [13]:
T_TEACHER = sc_pseudo_arr.shape[1]
assert T_TEACHER == xc_pseudo_arr.shape[1], f"T mismatch: SC={T_TEACHER} XC={xc_pseudo_arr.shape[1]}"
print(f"T_TEACHER: {T_TEACHER}, T_STUDENT: {T_STUDENT}")


def collate_fn(batch):
    wavs = torch.stack([b["wav"] for b in batch])
    clip_targets = torch.tensor([b["clip_target_idx"] for b in batch], dtype=torch.long)
    source_flags = torch.tensor([b["source_flag"] for b in batch], dtype=torch.long)
    pseudo_mask = source_flags > 0  # XC or SC
    if pseudo_mask.any():
        pseudos = torch.stack([b["frame_pseudo"] for b in batch if b["source_flag"] > 0])
    else:
        pseudos = torch.zeros(0, T_TEACHER, N_CLASSES, dtype=torch.float16)
    return {"wav": wavs, "clip_target": clip_targets, "source_flag": source_flags, "pseudos": pseudos}


train_loader = DataLoader(combined_train_ds, batch_size=CFG["batch_size"], shuffle=True,
                          num_workers=CFG["num_workers"], pin_memory=True, drop_last=True,
                          persistent_workers=True, collate_fn=collate_fn)
val_loader = DataLoader(bc_val_ds, batch_size=CFG["batch_size"], shuffle=False,
                        num_workers=CFG["num_workers"], pin_memory=True, persistent_workers=True,
                        collate_fn=lambda b: {"wav": torch.stack([x["wav"] for x in b]),
                                              "clip_target": torch.tensor([x["clip_target_idx"] for x in b], dtype=torch.long)})

n_steps = len(train_loader) * CFG["epochs"]
print(f"Steps/epoch: {len(train_loader)}, total: {n_steps}")

optimizer = optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])

def lr_lambda(step):
    if step < CFG["warmup_steps"]: return step / max(1, CFG["warmup_steps"])
    progress = (step - CFG["warmup_steps"]) / max(1, n_steps - CFG["warmup_steps"])
    return 0.5 * (1 + math.cos(math.pi * progress))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


T_TEACHER: 10, T_STUDENT: 10
Steps/epoch: 916, total: 13740


In [14]:
@torch.no_grad()
def evaluate():
    model.eval()
    all_logits = []; all_targets = []
    for batch in val_loader:
        wav = batch["wav"].to(DEVICE, non_blocking=True).unsqueeze(1)
        with autocast('cuda', dtype=torch.bfloat16):
            mel = mel_extractor(wav)
            mel = (mel - mel.mean(dim=(2, 3), keepdim=True)) / (mel.std(dim=(2, 3), keepdim=True) + 1e-6)
            logit = model(mel)
        all_logits.append(logit.float().cpu()); all_targets.append(batch["clip_target"])
    all_logits = torch.cat(all_logits); all_targets = torch.cat(all_targets)
    from sklearn.metrics import roc_auc_score
    probs = torch.sigmoid(all_logits).numpy()
    targets_onehot = np.eye(N_CLASSES)[all_targets.numpy()]
    aucs = []
    for c in range(N_CLASSES):
        if targets_onehot[:, c].sum() < 2: continue
        try: aucs.append(roc_auc_score(targets_onehot[:, c], probs[:, c]))
        except: pass
    aucs_arr = np.array(aucs)
    val_macro = float(np.mean(aucs_arr)) if len(aucs_arr) else 0.0
    ns_aucs = aucs_arr[aucs_arr < 1.0]
    val_ns22 = float(np.sort(ns_aucs)[:22].mean()) if len(ns_aucs) >= 22 else float(np.mean(ns_aucs)) if len(ns_aucs) > 0 else val_macro

    taxon_aucs = {}
    sp_df = species_df.copy(); sp_df["idx"] = sp_df["primary_label"].map(label_to_idx)
    for cls in sp_df["class_name"].unique():
        idx_list = sp_df[sp_df["class_name"] == cls]["idx"].tolist()
        cls_aucs = []
        for c in idx_list:
            if targets_onehot[:, c].sum() < 2: continue
            try: cls_aucs.append(roc_auc_score(targets_onehot[:, c], probs[:, c]))
            except: pass
        taxon_aucs[cls] = float(np.mean(cls_aucs)) if cls_aucs else float("nan")

    cstat = {
        "n": len(aucs_arr),
        "median": float(np.median(aucs_arr)) if len(aucs_arr) else 0.0,
        "p25": float(np.percentile(aucs_arr, 25)) if len(aucs_arr) else 0.0,
        "p75": float(np.percentile(aucs_arr, 75)) if len(aucs_arr) else 0.0,
        "n_gt05": int((aucs_arr > 0.5).sum()),
        "n_gt07": int((aucs_arr > 0.7).sum()),
        "n_gt09": int((aucs_arr > 0.9).sum()),
        "n_perfect": int((aucs_arr == 1.0).sum()),
    }
    return val_macro, val_ns22, taxon_aucs, cstat


In [15]:
best_val = 0.0
best_path = DRIVE_ROOT / "effv2b0_r2_sc_best.pth"
log_path = DRIVE_ROOT / "effv2b0_r2_sc_train.log"

step = 0
total_t0 = time.time()
for ep in range(CFG["epochs"]):
    t0 = time.time()
    model.train()
    losses = []; losses_bc = []; losses_pseudo = []
    n_bc_total = n_xc_total = n_sc_total = 0

    for bi, batch in enumerate(train_loader):
        wav = batch["wav"].to(DEVICE, non_blocking=True).unsqueeze(1)
        clip_target = batch["clip_target"].to(DEVICE)
        source_flag = batch["source_flag"].to(DEVICE)
        pseudos = batch["pseudos"].to(DEVICE, non_blocking=True).float()

        bc_mask = source_flag == 0
        xc_mask = source_flag == 1
        sc_mask = source_flag == 2
        n_bc = int(bc_mask.sum()); n_xc = int(xc_mask.sum()); n_sc = int(sc_mask.sum())
        n_bc_total += n_bc; n_xc_total += n_xc; n_sc_total += n_sc

        with autocast('cuda', dtype=torch.bfloat16):
            mel = mel_extractor(wav)
            mel = (mel - mel.mean(dim=(2, 3), keepdim=True)) / (mel.std(dim=(2, 3), keepdim=True) + 1e-6)
            mel = spec_augment(mel)
            clip_logits, frame_logits = model(mel, return_framewise=True)

            loss_total = 0.0
            loss_bc_val = torch.tensor(0.0, device=DEVICE)
            loss_pseudo_val = torch.tensor(0.0, device=DEVICE)

            # BC2026: clip-level BCE on hard label
            if n_bc > 0:
                bc_clip = clip_logits[bc_mask]
                bc_tgt_idx = clip_target[bc_mask]
                onh = F.one_hot(bc_tgt_idx, N_CLASSES).float()
                ls = CFG["label_smoothing"]
                smooth = onh * (1 - ls) + ls / N_CLASSES
                loss_bc_val = F.binary_cross_entropy_with_logits(bc_clip, smooth)
                loss_total = loss_total + loss_bc_val

            # XC + SC: frame-level BCE on pseudo (interpolate teacher T → student T)
            pseudo_mask = source_flag > 0
            n_pseudo = int(pseudo_mask.sum())
            if n_pseudo > 0:
                p_frame = frame_logits[pseudo_mask]
                pp = pseudos.permute(0, 2, 1)
                pp = F.interpolate(pp, size=T_STUDENT, mode="linear", align_corners=False)
                pp = pp.permute(0, 2, 1).clamp(min=1e-6, max=1.0 - 1e-6)
                # XC weight 0.5、SC weight 0.5 (uniform on pseudo samples)
                w_pseudo = CFG["xc_pseudo_weight"]  # = sc_pseudo_weight same value 0.5
                loss_pseudo_val = F.binary_cross_entropy_with_logits(p_frame, pp)
                loss_total = loss_total + w_pseudo * loss_pseudo_val

            loss = loss_total / max(1.0, float((n_bc > 0) + (n_pseudo > 0)) * 0.5)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
        optimizer.step()
        scheduler.step()
        losses.append(float(loss.item()))
        if n_bc > 0: losses_bc.append(float(loss_bc_val.item()))
        if n_pseudo > 0: losses_pseudo.append(float(loss_pseudo_val.item()))
        step += 1
        if bi % 100 == 0:
            print(f"  [ep{ep+1} step {bi}/{len(train_loader)}] loss={loss.item():.4f} bc={loss_bc_val.item():.4f} pseudo={loss_pseudo_val.item():.4f} n_bc={n_bc} n_xc={n_xc} n_sc={n_sc} lr={scheduler.get_last_lr()[0]:.2e}")

    avg_loss = float(np.mean(losses))
    avg_bc = float(np.mean(losses_bc)) if losses_bc else 0.0
    avg_pseudo = float(np.mean(losses_pseudo)) if losses_pseudo else 0.0
    val_macro, val_ns22, taxon_aucs, cstat = evaluate()
    ep_el = time.time() - t0
    tot_el = time.time() - total_t0
    cur_lr = scheduler.get_last_lr()[0]
    is_best = val_ns22 > best_val
    best_mark = "BEST" if is_best else ""

    l1 = (f"=== Ep {ep+1}/{CFG['epochs']}: loss={avg_loss:.4f} (bc={avg_bc:.4f} pseudo={avg_pseudo:.4f}) "
          f"val_ns22={val_ns22:.4f} val_macro={val_macro:.4f} {best_mark} lr={cur_lr:.2e} "
          f"({ep_el/60:.1f}min, total {tot_el/60:.1f}min) n_bc={n_bc_total} n_xc={n_xc_total} n_sc={n_sc_total} ===")
    l2 = "    taxon: " + " ".join(f"{k}={v:.3f}" if v == v else f"{k}=nan" for k, v in taxon_aucs.items())
    l3 = (f"    class: n={cstat['n']} median={cstat['median']:.3f} p25={cstat['p25']:.3f} p75={cstat['p75']:.3f} "
          f"#>0.5={cstat['n_gt05']} #>0.7={cstat['n_gt07']} #>0.9={cstat['n_gt09']} #perfect={cstat['n_perfect']}")
    print(l1); print(l2); print(l3)
    with open(log_path, "a") as f:
        f.write(l1 + "\n" + l2 + "\n" + l3 + "\n")

    if is_best:
        best_val = val_ns22
        torch.save({"state_dict": model.state_dict(), "val_ns22": val_ns22, "val_macro": val_macro, "ep": ep+1, "cfg": CFG}, best_path)
        print(f"    BEST saved val_ns22={val_ns22:.4f}")

print(f"\nDone. Best val_ns22: {best_val:.4f}")
print(f"Best ckpt: {best_path}")


  [ep1 step 0/916] loss=0.0204 bc=0.0132 pseudo=0.0144 n_bc=28 n_xc=93 n_sc=71 lr=1.33e-06
  [ep1 step 100/916] loss=0.0179 bc=0.0118 pseudo=0.0122 n_bc=32 n_xc=88 n_sc=72 lr=1.35e-04
  [ep1 step 200/916] loss=0.0180 bc=0.0127 pseudo=0.0106 n_bc=41 n_xc=85 n_sc=66 lr=2.68e-04
  [ep1 step 300/916] loss=0.0194 bc=0.0137 pseudo=0.0114 n_bc=30 n_xc=95 n_sc=67 lr=4.00e-04
  [ep1 step 400/916] loss=0.0172 bc=0.0114 pseudo=0.0117 n_bc=27 n_xc=102 n_sc=63 lr=4.00e-04
  [ep1 step 500/916] loss=0.0176 bc=0.0120 pseudo=0.0110 n_bc=33 n_xc=97 n_sc=62 lr=4.00e-04
  [ep1 step 600/916] loss=0.0189 bc=0.0136 pseudo=0.0108 n_bc=34 n_xc=85 n_sc=73 lr=4.00e-04
  [ep1 step 700/916] loss=0.0173 bc=0.0118 pseudo=0.0111 n_bc=31 n_xc=89 n_sc=72 lr=3.99e-04
  [ep1 step 800/916] loss=0.0174 bc=0.0120 pseudo=0.0110 n_bc=35 n_xc=91 n_sc=66 lr=3.99e-04
  [ep1 step 900/916] loss=0.0168 bc=0.0116 pseudo=0.0103 n_bc=40 n_xc=87 n_sc=65 lr=3.98e-04
=== Ep 1/15: loss=0.0180 (bc=0.0123 pseudo=0.0114) val_ns22=0.7690 val_

KeyboardInterrupt: 

In [ ]:
import os, json, shutil
USER = "maekeso"
SLUG = "birdclef2026-exp065-effv2b0-r2-sc"
TITLE = "BirdCLEF2026 exp065 effv2b0 R2 SC"

UPLOAD_DIR = Path("/content/upload_exp065")
UPLOAD_DIR.mkdir(exist_ok=True, parents=True)
shutil.copy(best_path, UPLOAD_DIR / "effv2b0_r2_sc_best.pth")
shutil.copy(log_path, UPLOAD_DIR / "effv2b0_r2_sc_train.log")

meta = {"title": TITLE, "id": f"{USER}/{SLUG}", "licenses": [{"name":"other"}]}
(UPLOAD_DIR / "dataset-metadata.json").write_text(json.dumps(meta, indent=2))

from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()
try:
    api.dataset_create_new(folder=str(UPLOAD_DIR), public=False, dir_mode="zip", quiet=False)
    print("OK new dataset")
except Exception as e:
    print(f"create_new err: {str(e)[:200]}")
    try:
        api.dataset_create_version(folder=str(UPLOAD_DIR), version_notes="exp065 R2 SC", dir_mode="zip", quiet=False)
        print("OK version")
    except Exception as e2:
        print(f"err: {str(e2)[:200]}")
print(f"URL: https://www.kaggle.com/datasets/{USER}/{SLUG}")


In [ ]:
from google.colab import runtime
runtime.unassign()
